# Multi-Agent Care Pathway Coordination (Strands + AgentCore Memory + Metadata)

## Introduction

This tutorial builds a **multi-agent healthcare assistant** — intake bot, nurse triage, and cardiology specialist — that share a single patient-scoped memory namespace. The metadata schema makes that shared memory *usable* across agents:

- **Provenance**: every memory is tagged with the agent that wrote it (`source_agent`).
- **Clinical reasoning trails**: ICD-10 codes in a STRINGLIST let you ask *"when was PE considered for this patient?"*
- **Workflow state**: `care_pathway_stage` uses a **monotonic lifecycle merge** — once a patient reaches the `treatment` stage, consolidation never regresses back to `intake`.

The custom merge rules are the distinctive thing here — they go beyond `LATEST_VALUE` to encode domain logic (specialist hierarchy, ordinal workflow progression) directly into the extraction pipeline.

### Tutorial Details

| Information         | Details                                                                           |
|:--------------------|:----------------------------------------------------------------------------------|
| Tutorial type       | Long-term memory with metadata filtering (multi-agent)                            |
| Agent type          | Healthcare care-pathway (supervisor + intake bot + nurse triage + cardiology)     |
| Agentic Framework   | Strands Agents                                                                    |
| LLM model           | Anthropic Claude Haiku 4.5                                                        |
| Tutorial components | Semantic strategy, custom merge rules, ICD-10 STRINGLIST, batch ingestion         |
| Example complexity  | Advanced                                                                          |

### You'll learn to
- Coordinate three sub-agents writing into one shared namespace via metadata provenance
- Author **custom `llmExtractionInstruction`** beyond `LATEST_VALUE`: specialist-hierarchy merge and monotonic lifecycle merge
- Use STRINGLIST `CONTAINS` on ICD-10 codes for clinical-reasoning retrieval
- Use `batch_create_memory_records` — with and without `memoryStrategyId` — to show both modes of direct ingestion

## Prerequisites
- Python 3.10+
- AWS credentials with `bedrock-agentcore` and `bedrock-agentcore-control` permissions
- A `memory_execution_role_arn`
- Amazon Bedrock access to Anthropic Claude Haiku 4.5


## Step 1: Install Dependencies

In [ ]:
!pip install -qr requirements.txt

## Step 2: Imports and Configuration

In [ ]:
import logging
import time
import json
import uuid
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional

import boto3
from botocore.exceptions import ClientError

from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

from bedrock_agentcore.memory import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.models import StringValue

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("care-pathway-metadata")

USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT
logger.info("Imports loaded")


In [ ]:
# Replace with your values
REGION = "us-west-2"
MEMORY_EXECUTION_ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<AgentCoreMemoryExecutionRole>"

PATIENT_ID = "patient-0042"
SESSION_ID = f"visit_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info(f"Region:  {REGION}")
logger.info(f"Patient: {PATIENT_ID}")
logger.info(f"Session: {SESSION_ID}")


## Step 3: Create Memory with Custom Merge Rules

This is the schema that does the interesting work. Two fields use **non-default merge semantics**:

**`source_agent` — specialist hierarchy.** When multiple agents contribute to the same consolidated memory, the merged record reflects the **most senior** handler rather than the most recent one. The FM reads the instruction literally:

> *"Retain the most senior handler. Hierarchy: specialist > nurse_triage > intake_bot. Never downgrade."*

**`care_pathway_stage` — monotonic workflow advance.** The stage only moves forward. If the consolidated memory was touched at `assessment` and then later at `treatment`, the merged value is `treatment`. A follow-up conversation that *mentions* intake notes does not regress the tag.

> *"Advance to the latest stage reached. Never regress even if a later event references an earlier one."*


In [ ]:
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

memory_name = "CarePathwayMultiAgentMemory"

indexed_keys = [
    {"key": "source_agent",          "type": "STRING"},
    {"key": "clinical_acuity",       "type": "NUMBER"},
    {"key": "differential_dx_codes", "type": "STRING_LIST"},
    {"key": "care_pathway_stage",    "type": "STRING"},
    # billing_relevant is intentionally NOT indexed
]

PATHWAY_STAGES = ["intake", "assessment", "diagnostic_workup", "treatment", "follow_up", "discharge"]

metadata_schema = [
    {
        "key": "source_agent",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "The clinical role that authored or most-authoritatively shaped the memory. "
                    "Values: intake_bot, nurse_triage, cardiology_agent, primary_care_agent."
                ),
                "llmExtractionInstruction": (
                    "When multiple agents contribute to the same memory, retain the most senior handler. "
                    "Hierarchy (most to least senior): cardiology_agent > primary_care_agent > nurse_triage > intake_bot. "
                    "Never downgrade from a specialist to a lower tier, even if the lower tier was the most recent to write."
                ),
                "validation": {
                    "stringValidation": {
                        "allowedValues": ["intake_bot", "nurse_triage", "cardiology_agent", "primary_care_agent"]
                    }
                }
            }
        }
    },
    {
        "key": "clinical_acuity",
        "type": "NUMBER",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Emergency Severity Index (ESI) 1-5. 1 = resuscitation, 2 = emergent, 3 = urgent, "
                    "4 = less urgent, 5 = non-urgent. LOWER IS WORSE."
                ),
                "llmExtractionInstruction": (
                    "Infer from vitals, presenting complaint, and stated urgency. Default to the worst "
                    "(lowest) ESI referenced across the conversation."
                ),
                "validation": {
                    "numericValidation": {"minimum": 1, "maximum": 5}
                }
            }
        }
    },
    {
        "key": "differential_dx_codes",
        "type": "STRING_LIST",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "ICD-10 codes corresponding to diagnoses actively being considered "
                    "in the conversation's differential. Examples: I20.9 angina, I26.99 pulmonary embolism, "
                    "R07.9 chest pain unspecified."
                ),
                "llmExtractionInstruction": (
                    "Extract every ICD-10 code that is being actively considered as a potential diagnosis "
                    "(not ones that are definitively ruled out). Return the codes as a list. "
                    "Use standard ICD-10 format (letter + digits + optional decimal)."
                )
            }
        }
    },
    {
        "key": "care_pathway_stage",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Current stage of the patient's care pathway. Pathway order: "
                    "intake -> assessment -> diagnostic_workup -> treatment -> follow_up -> discharge."
                ),
                "llmExtractionInstruction": (
                    "Stages progress in this strict order: intake, assessment, diagnostic_workup, "
                    "treatment, follow_up, discharge. During consolidation, set the value to the LATEST "
                    "stage reached in pathway order. NEVER regress to an earlier stage even if a later "
                    "event mentions an earlier one (e.g. a follow-up discussion that references intake "
                    "notes must stay at follow_up)."
                ),
                "validation": {
                    "stringValidation": {
                        "allowedValues": PATHWAY_STAGES
                    }
                }
            }
        }
    },
    {
        "key": "billing_relevant",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Whether this memory references a chargeable clinical event. Values: yes, no."
                ),
                "llmExtractionInstruction": "yes if any billable procedure/test/visit is referenced; no otherwise.",
                "validation": {"stringValidation": {"allowedValues": ["yes", "no"]}}
            }
        }
    },
]

try:
    resp = control_client.create_memory(
        name=memory_name,
        eventExpiryDuration=90,
        memoryExecutionRoleArn=MEMORY_EXECUTION_ROLE_ARN,
        indexedKeys=indexed_keys,
        memoryStrategies=[{
            "semanticMemoryStrategy": {
                "name": "CarePathwaySemantic",
                "description": "Patient memories with provenance, clinical, and workflow metadata",
                "namespaceTemplates": ["/patients/{actorId}/"],
                "memoryRecordSchema": {"metadataSchema": metadata_schema}
            }
        }],
        clientToken=str(uuid.uuid4()),
    )
    memory_id = resp["memory"]["id"]
    logger.info(f"✅ Created memory {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        existing = control_client.list_memories()["memories"]
        match = next((m for m in existing if m["name"] == memory_name), None)
        memory_id = match["id"]
        logger.info(f"ℹ️  Reusing existing memory {memory_id}")
    else:
        raise

strategy_id = control_client.get_memory(memoryId=memory_id)["memory"]["strategies"][0]["strategyId"]
namespace = f"/patients/{PATIENT_ID}/"
logger.info(f"Strategy id: {strategy_id}")
logger.info(f"Namespace:   {namespace}")


## Step 4: Wait for Memory to be Active


In [ ]:
def wait_active(mem_id, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        m = control_client.get_memory(memoryId=mem_id)["memory"]
        logger.info(f"status: {m['status']}")
        if m["status"] == "ACTIVE":
            return
        if m["status"] == "FAILED":
            raise RuntimeError(m.get("failureReason"))
        time.sleep(10)
    raise TimeoutError("memory never reached ACTIVE")

wait_active(memory_id)


## Step 5: Multi-Agent Hook Provider

All three sub-agents share the same patient namespace but each has its own hook provider that stamps outgoing events with its **own `source_agent`** and the **stage it's operating in**. The stage and specialist-hierarchy merge rules are what make this coordination work.


In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=REGION)
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
patient_session = session_manager.create_memory_session(actor_id=PATIENT_ID, session_id=SESSION_ID)

class AgentProvenanceHooks(HookProvider):
    def __init__(self, session, source_agent: str, care_stage: str):
        self.session = session
        self.source_agent = source_agent
        self.care_stage = care_stage

    def save_turn(self, event: AfterInvocationEvent):
        msgs = event.agent.messages
        if len(msgs) < 2 or msgs[-1]["role"] != "assistant":
            return
        user_text = next((m["content"][0].get("text", "") for m in reversed(msgs) if m["role"] == "user"), "")
        asst_text = msgs[-1]["content"][0].get("text", "")
        if not user_text or not asst_text:
            return
        try:
            self.session.add_turns(
                messages=[ConversationalMessage(user_text, USER), ConversationalMessage(asst_text, ASSISTANT)],
                metadata={
                    "source_agent":       StringValue.build(self.source_agent),
                    "care_pathway_stage": StringValue.build(self.care_stage),
                }
            )
            logger.info(f"[{self.source_agent}] saved at stage={self.care_stage}")
        except Exception as e:
            logger.error(f"save failed: {e}")

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AfterInvocationEvent, self.save_turn)

logger.info("Hook provider ready")


## Step 6: Seed a Patient Journey

We simulate a realistic patient journey end-to-end:
1. **Intake bot** (stage = `intake`): collects presenting complaint and vitals.
2. **Nurse triage** (stage = `assessment`): assigns ESI and flags the working differential.
3. **Cardiology agent** (stage = `diagnostic_workup` → `treatment`): conducts workup and commits treatment.

All three write into `patients/patient-0042`. No event carries `clinical_acuity`, `differential_dx_codes`, or `billing_relevant` — those come from the FM. The two custom merge rules should collapse the consolidated memory onto the specialist agent, at the latest pathway stage.


In [ ]:
def submit(role_agent: str, stage: str, turns):
    hook = AgentProvenanceHooks(patient_session, source_agent=role_agent, care_stage=stage)
    # Write directly rather than via a dummy Strands agent:
    for i, (role, text) in enumerate(turns):
        msg = ConversationalMessage(text, USER if role == "user" else ASSISTANT)
        patient_session.add_turns(
            messages=[msg],
            metadata={
                "source_agent":       StringValue.build(role_agent),
                "care_pathway_stage": StringValue.build(stage),
            }
        )
    logger.info(f"seeded {role_agent} @ {stage} ({len(turns)} turns)")

# 1. Intake bot collects presenting complaint
submit("intake_bot", "intake", [
    ("user",      "I have chest pain that started this morning. Sharp, radiates to my left arm."),
    ("assistant", "I've recorded: chest pain with left-arm radiation, onset this morning. Vitals please: BP 156/94, HR 102, SpO2 96%. Escalating to triage."),
])

# 2. Nurse triage assigns ESI and flags differential
submit("nurse_triage", "assessment", [
    ("user",      "Patient reports sharp substernal chest pain, radiating L arm, onset ~2h ago, BP 156/94, HR 102."),
    ("assistant", "ESI 2 — emergent. Differential includes angina (I20.9), acute MI (I21.9), and given HR elevation we can't rule out PE (I26.99). Calling cardiology."),
])

# 3. Cardiology agent does workup -> treatment
submit("cardiology_agent", "diagnostic_workup", [
    ("user",      "Cards here. ECG shows ST changes in lateral leads, troponin mildly elevated. Patient reports no DVT symptoms, no recent travel."),
    ("assistant", "ECG + troponin trajectory argue against PE. More consistent with NSTEMI vs unstable angina. Ordering serial troponins and coronary angiography."),
])

submit("cardiology_agent", "treatment", [
    ("user",      "Angio shows 80% LAD stenosis. Plan: PCI with drug-eluting stent."),
    ("assistant", "Treatment committed: PCI with DES to LAD. Dual antiplatelet therapy initiated. Clinical improvement expected within 24h."),
])

logger.info("Seeded patient journey across 3 agents and 4 pathway stages")


## Step 7: Wait for Extraction


In [ ]:
def wait_records(expected_min=2, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        r = data_client.list_memory_records(memoryId=memory_id, namespace=namespace, maxResults=50)
        recs = r.get("memoryRecordSummaries", [])
        logger.info(f"records so far: {len(recs)}")
        if len(recs) >= expected_min:
            return recs
        time.sleep(15)
    raise TimeoutError(f"only {len(recs)} records")

records = wait_records(expected_min=2)
logger.info(f"✅ {len(records)} records extracted")


## Step 8: Verify Both Custom Merges

Print every consolidated record. Expect to see:
- Records touched by both triage and cardiology → `source_agent = cardiology_agent` (specialist-hierarchy merge).
- Records spanning multiple stages → `care_pathway_stage = treatment` (monotonic merge).
- `differential_dx_codes` populated with the ICD-10 codes from the conversation — even though no event supplied them.


In [ ]:
for i, r in enumerate(records, 1):
    full = data_client.get_memory_record(memoryId=memory_id, memoryRecordId=r["memoryRecordId"])["memoryRecord"]
    print(f"\n=== Record #{i} ===")
    print(f"content: {full.get('content', {}).get('text', '')[:180]}")
    for k, v in full.get("metadata", {}).items():
        print(f"  {k}: {v}")


## Step 9: Retrieval — Multi-Agent Queries

### Query 1 — provenance filter

*"What did cardiology conclude for this patient?"* — filter by `source_agent = cardiology_agent`.


In [ ]:
def retrieve(query, filters=None, top_k=10):
    search = {"searchQuery": query, "topK": top_k}
    if filters:
        search["metadataFilters"] = filters
    r = data_client.retrieve_memory_records(
        memoryId=memory_id, namespace=namespace, searchCriteria=search
    )
    return r.get("memoryRecordSummaries", [])

def show(results, label):
    print(f"\n--- {label}: {len(results)} results ---")
    for r in results:
        print(f"  [{r.get('score', 0):.3f}] {r.get('content', {}).get('text', '')[:160]}")

cards_only = retrieve(
    "clinical findings",
    filters=[
        {"left": {"metadataKey": "source_agent"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "cardiology_agent"}}},
    ],
)
show(cards_only, "source_agent = cardiology_agent")


### Query 2 — clinical reasoning trail (ICD-10 CONTAINS)

*"Every memory where PE (I26.99) was on the differential."*


In [ ]:
pe_considered = retrieve(
    "chest pain",
    filters=[
        {"left": {"metadataKey": "differential_dx_codes"}, "operator": "CONTAINS",
         "right": {"metadataValue": {"stringValue": "I26.99"}}},
    ],
)
show(pe_considered, "differential_dx_codes CONTAINS 'I26.99' (pulmonary embolism)")


### Query 3 — workflow stage filter

*"Treatment-stage findings only — no intake, no assessment."*


In [ ]:
tx_stage = retrieve(
    "what was the plan",
    filters=[
        {"left": {"metadataKey": "care_pathway_stage"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "treatment"}}},
    ],
)
show(tx_stage, "care_pathway_stage = treatment")


### Query 4 — compound filter

*"When cardiology considered angina in the treatment stage."* Three filters AND'd.


In [ ]:
compound = retrieve(
    "what happened",
    filters=[
        {"left": {"metadataKey": "source_agent"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "cardiology_agent"}}},
        {"left": {"metadataKey": "differential_dx_codes"}, "operator": "CONTAINS",
         "right": {"metadataValue": {"stringValue": "I20.9"}}},
        {"left": {"metadataKey": "care_pathway_stage"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "treatment"}}},
    ],
)
show(compound, "cardiology_agent AND I20.9 (angina) AND treatment")


### Query 5 — acuity band

*"High-acuity encounters only (ESI ≤ 2)."*


In [ ]:
high_acuity = retrieve(
    "emergent",
    filters=[
        {"left": {"metadataKey": "clinical_acuity"}, "operator": "LESS_THAN_OR_EQUALS",
         "right": {"metadataValue": {"numberValue": 2}}},
    ],
)
show(high_acuity, "clinical_acuity <= 2 (ESI 1-2, emergent)")


## Step 10: `batch_create_memory_records` — Both Modes

The batch API lets you bypass event-driven extraction and write records directly. The behavior changes dramatically depending on whether you supply `memoryStrategyId`:

- **With `memoryStrategyId`** — the service filters input metadata to only keys defined in that strategy's schema. Non-schema keys are **silently dropped**.
- **Without `memoryStrategyId`** — the service stores all metadata keys as-is. Non-indexed keys are stored but not filterable.

Both are demonstrated below.


In [ ]:
# Mode 1: WITH memoryStrategyId — schema-enforced consistency
r1 = data_client.batch_create_memory_records(
    memoryId=memory_id,
    records=[{
        "requestIdentifier": f"ref-{uuid.uuid4()}",
        "namespaces": [namespace],
        "content": {"text": "Reference note: patient has documented family history of early MI."},
        "timestamp": datetime.now(timezone.utc),
        "memoryStrategyId": strategy_id,
        "metadata": {
            "source_agent":       {"stringValue": "cardiology_agent"},
            "care_pathway_stage": {"stringValue": "assessment"},
            # The next key IS indexed but NOT in the strategy schema (we didn't add it).
            # Actually all our indexed keys ARE in the schema, so this example is clean.
            # Let's include a key that is NEITHER indexed nor in schema:
            "unrelated_tag": {"stringValue": "this-will-be-dropped"},
        },
    }],
)
print("With memoryStrategyId (schema-enforced):")
print(json.dumps(r1.get("successfulRecords", []), default=str, indent=2)[:400])


In [ ]:
# Mode 2: WITHOUT memoryStrategyId — all keys stored, non-indexed not filterable
r2 = data_client.batch_create_memory_records(
    memoryId=memory_id,
    records=[{
        "requestIdentifier": f"import-{uuid.uuid4()}",
        "namespaces": [namespace],
        "content": {"text": "Imported from legacy EMR: patient has type 2 diabetes, well controlled."},
        "timestamp": datetime.now(timezone.utc),
        "metadata": {
            "source_agent":       {"stringValue": "primary_care_agent"},
            "care_pathway_stage": {"stringValue": "assessment"},
            # Non-schema keys are stored as-is without memoryStrategyId:
            "legacy_system":      {"stringValue": "mrn_v1"},
            "import_batch":       {"stringValue": "2026-04-30-nightly"},
        },
    }],
)
print("Without memoryStrategyId (pass-through):")
print(json.dumps(r2.get("successfulRecords", []), default=str, indent=2)[:400])


## Step 11: Supervisor Agent — Routes by Metadata

The supervisor answers a question about the patient by retrieving cardiology-only memories first, then primary-care-only, then combining.


In [ ]:
@tool
def retrieve_for_agent_role(query: str, source_agent: str, max_results: int = 5) -> str:
    """Retrieve memories authored by a specific agent role.

    Args:
        query: free-text query for semantic relevance within the filtered set.
        source_agent: one of intake_bot, nurse_triage, cardiology_agent, primary_care_agent.
        max_results: maximum number of results.
    """
    recs = retrieve(query, filters=[
        {"left": {"metadataKey": "source_agent"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": source_agent}}},
    ], top_k=max_results)
    if not recs:
        return f"no memories from {source_agent}"
    return "\n".join(f"- {r.get('content', {}).get('text', '')[:240]}" for r in recs)


@tool
def retrieve_by_dx_code(query: str, icd10: str) -> str:
    """Retrieve memories where a specific ICD-10 code was on the differential."""
    recs = retrieve(query, filters=[
        {"left": {"metadataKey": "differential_dx_codes"}, "operator": "CONTAINS",
         "right": {"metadataValue": {"stringValue": icd10}}},
    ], top_k=5)
    if not recs:
        return f"no memories where {icd10} was considered"
    return "\n".join(f"- {r.get('content', {}).get('text', '')[:240]}" for r in recs)


supervisor = Agent(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[retrieve_for_agent_role, retrieve_by_dx_code],
    system_prompt=(
        "You are a care coordination supervisor. To answer questions about a patient's care, "
        "you MUST retrieve memories filtered by the appropriate agent role or diagnosis code — "
        "never retrieve across all agents indiscriminately. When a question is about clinical findings, "
        "filter by source_agent='cardiology_agent' or 'primary_care_agent'. When a question is about a "
        "specific diagnosis, use retrieve_by_dx_code with the ICD-10 code."
    ),
)

question = "What was cardiology's final treatment plan for the patient's chest pain?"
print("QUESTION:", question)
print()
print("ANSWER:\n", supervisor(question))


## Step 12: Cleanup (Optional)


In [ ]:
# control_client.delete_memory(memoryId=memory_id)
# print(f"Deleted {memory_id}")


## What you built

- Three sub-agents writing into **one shared patient namespace** with provenance metadata (`source_agent`) — no need for separate memory resources per role.
- Two **custom `llmExtractionInstruction`** rules that go beyond `LATEST_VALUE`:
  - **Specialist-hierarchy merge** — consolidation retains the most senior author.
  - **Monotonic lifecycle merge** — `care_pathway_stage` only advances, never regresses.
- **ICD-10 STRINGLIST retrieval** — `differential_dx_codes CONTAINS "I26.99"` surfaces every memory where PE was considered, implementing a clinical-reasoning trail.
- **Compound retrieval** — provenance + diagnosis code + stage, all AND'd in one query.
- **Both `batch_create_memory_records` modes** — schema-enforced (with `memoryStrategyId`) and pass-through (without), for data-import scenarios.

### Takeaways
- Metadata makes a *shared* memory layer usable across agents; without it, multi-agent coordination tends to require separate stores per role.
- Custom merge instructions encode domain invariants (clinical authority, workflow progression) directly into the extraction pipeline.
- Non-indexed schema keys enrich records without burning the 10-key budget.
- `batch_create_memory_records` behavior depends critically on `memoryStrategyId` — be explicit about which mode you want.
